<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/P06/MSA0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alinhamento Múltiplo de Sequências (MSA)

O alinhamento múltiplo de sequências é uma importante ferramenta para a caracterização de regiões conservadas em proteínas de uma família. Este tipo de alinhamento permite a comparação simultânea de sequências, fazendo com que a predição de conservação de resíduos seja muito mais confiável. Além disso, o alinhamento múltiplo permite a realização de análises filogenéticas de famílias de proteínas (assunto a ser abordado na próxima aula), que fornecem informações sobre a sua evolução.

## 1. Alinhamento Progressivo com ClustalW

A primeira atividade utiliza um conjunto de sequências de proteínas humanas obtidas a partir do UniProt.

Realizaremos o alinhamento múltiplo dessas sequências usando o programa **ClustalW**.

A biblioteca Biopython não inclui o ClustalW, mas contém ferramentas para ler os arquivos que ele produz (o alinhamento e a árvore-guia). Vamos executar o ClustalW diretamente e usar o Biopython para analisar os resultados.

### Preparando o Ambiente para a Execução

O código abaixo instala o Biopython no ambiente do Google Colab e importa as bibliotecas necessárias.

Na sequência, o código faz o download do ClustalW a partir [deste link](http://www.clustal.org/download/current/clustalw-2.1-linux-x86_64-libcppstatic.tar.gz). O download só é feito se o programa ainda não estiver na pasta, então a célula pode ser executada mais de uma vez sem problemas.

In [ ]:
#@title Preparando o Ambiente para a Execução

# Instalando o Biopython e importando as bibliotecas necessárias...

%pip install -q biopython
import os
import re
import subprocess
from Bio import SeqIO, AlignIO, Phylo

# Baixando o ClustalW2 (apenas se ainda não estiver disponível)...

if not os.path.isfile("clustalw2"):
    !wget -q -nc http://www.clustal.org/download/current/clustalw-2.1-linux-x86_64-libcppstatic.tar.gz
    !tar -xzf clustalw-2.1-linux-x86_64-libcppstatic.tar.gz
    !mv clustalw-2.1-linux-x86_64-libcppstatic/clustalw2 .

assert os.path.isfile("clustalw2"), "Executável do ClustalW não encontrado"
print("Ambiente pronto!")

### Upload das Sequências que Serão Alinhadas

Nesta seção, faremos o download das sequências das proteínas que serão alinhadas. Neste notebook, assumimos que as sequências estão em formato **fasta**.

O arquivo que usaremos está disponível [aqui](https://github.com/grsart/BiomolComp/blob/main/Pratica5/Seqs0.fasta). Clique no link e abra em uma nova aba para visualizar as sequências.

**Q1 - Consegue antecipar alguma informação sobre como vai ser o alinhamento? O que pode falar sobre os organismos e sobre os tamanhos das sequências?**

In [ ]:
seq_file = "Seqs0.fasta"
!wget -q -nc https://raw.githubusercontent.com/grsart/BiomolComp/refs/heads/main/Pratica5/Seqs0.fasta
!head {seq_file}

### Inspecionando as Sequências que Serão Alinhadas

Nesta próxima seção, vamos inspecionar o nosso arquivo fasta, que contém múltiplas sequências. Vamos usar as funções do Biopython (SeqIO) para isto.

Cada sequência é lida pela função *SeqIO.parse*. Dela, podemos listar a própria sequência, o cabeçalho (Id) e o número de resíduos, por exemplo.

In [ ]:
count = 0
for seq_record in SeqIO.parse(seq_file, "fasta"):
    count = count + 1
    print("SeqID: %s" % seq_record.id)
    print(repr(seq_record.seq))
    print("Sequence Length: %d" % len(seq_record))
print("")
print("Number of sequences found: %d" % count)

### Alinhamento das Sequências com o ClustalW

Nesta seção, faremos o alinhamento pelo método progressivo, usando o software *ClustalW*.

No método progressivo, as sequências são inicialmente alinhadas par a par e uma **distância** entre as sequências é calculada com base no score de cada alinhamento (maior score significa maior similaridade e menor distância entre as sequências). A partir dessas distâncias é construída uma **árvore-guia**. Em seguida, o alinhamento múltiplo é construído pela adição progressiva das sequências, seguindo a árvore-guia e começando pelas mais próximas.

O programa [ClustalW](http://www.clustal.org/clustal2/) foi descontinuado, mas ainda pode ser obtido. Em seu lugar, foi lançado o programa [Clustal Omega](http://www.clustal.org/omega/), que emprega modelos ocultos de Markov (HMM) para um alinhamento mais rápido e preciso, particularmente quando o número de sequências a ser alinhado é grande.

Na célula abaixo, definimos duas funções auxiliares: uma executa o `clustalw2` (que já baixamos e deixamos na pasta local) com o arquivo fasta como entrada, e a outra extrai, da saída do programa, os **scores dos alinhamentos par a par** (a primeira etapa do método progressivo). Guarde esses números: eles serão úteis nas questões seguintes.

In [ ]:
def rodar_clustalw(arquivo_fasta):
    """Executa o ClustalW e devolve (arquivo .aln, arquivo .dnd, saída do programa)."""
    resultado = subprocess.run(["./clustalw2", f"-INFILE={arquivo_fasta}"],
                               capture_output=True, text=True, check=True)
    base = os.path.splitext(arquivo_fasta)[0]
    return base + ".aln", base + ".dnd", resultado.stdout


def scores_par_a_par(saida):
    """Extrai da saída do ClustalW o score de cada alinhamento par a par."""
    padrao = r"Sequences \((\d+):(\d+)\) Aligned\. Score:\s*([\d.]+)"
    return {(int(i), int(j)): float(s) for i, j, s in re.findall(padrao, saida)}


def mostrar_scores(arquivo_fasta, saida):
    nomes = [rec.id for rec in SeqIO.parse(arquivo_fasta, "fasta")]
    scores = scores_par_a_par(saida)
    if not scores:
        print(saida)
        return
    print("Score  Par de sequências")
    for (i, j), score in scores.items():
        print(f"{score:5.0f}  {nomes[i - 1]}  x  {nomes[j - 1]}")


aln_file, dnd_file, saida = rodar_clustalw(seq_file)
mostrar_scores(seq_file, saida)

### Inspecionando o Alinhamento

Com sorte, o alinhamento já foi realizado pelo ClustalW. Dois arquivos devem ter sido escritos como saída do ClustalW: um arquivo com extensão .aln e um arquivo com extensão .dnd.

O primeiro arquivo contém o MSA. O segundo contém a **árvore-guia**, construída a partir das distâncias entre as sequências, no formato Newick. Vamos analisar os dois.

Primeiramente, vamos exibir o arquivo .aln. Abaixo de cada bloco do alinhamento, o ClustalW escreve uma linha de consenso:

* `*` coluna com o mesmo resíduo em todas as sequências;
* `:` substituições conservativas (propriedades fortemente semelhantes);
* `.` substituições semiconservativas (propriedades fracamente semelhantes).

In [ ]:
!cat {aln_file}

Também podemos usar o *parser* do Biopython (AlignIO) para abrir o arquivo .aln e fazer contas com o alinhamento, como contar quantas colunas são idênticas em todas as sequências.

In [ ]:
align = AlignIO.read(aln_file, "clustal")
n_colunas = align.get_alignment_length()
identicas = sum(1 for k in range(n_colunas) if len(set(align[:, k])) == 1)
print(f"{len(align)} sequências, {n_colunas} colunas no alinhamento")
print(f"Colunas idênticas em todas as sequências: {identicas}")
for record in align:
    gaps = record.seq.count("-")
    print(f"{record.id:30s} {gaps:4d} gaps ({100 * gaps / n_colunas:.0f}% do alinhamento)")

#### Visualização colorida (opcional)

A biblioteca `pymsaviz` desenha o alinhamento com os resíduos coloridos, o que facilita encontrar blocos conservados e regiões cheias de gaps.

In [ ]:
%pip install -q pymsaviz
import matplotlib.pyplot as plt
from IPython.display import Image
from pymsaviz import MsaViz

AlignIO.write(align, "Seqs0_alinhado.fasta", "fasta")
mv = MsaViz("Seqs0_alinhado.fasta", wrap_length=80, show_count=True, show_consensus=True)
mv.savefig("Seqs0_alinhamento.png")
plt.close("all")
Image("Seqs0_alinhamento.png")

### Questão 2

**A) Como ficou o alinhamento? Ele está confiável?**

**B) Existem regiões conservadas? E regiões mal alinhadas?**

**C) Quais são os aspectos das sequências de entrada que explicam esse comportamento?**

**D) Por que a ANX11_HUMAN produz um longo trecho sem correspondência no início do alinhamento? Observe a composição dos seus primeiros ~200 resíduos.**

### Analisando as Distâncias entre as Sequências

Uma vez que usamos um método progressivo e que as distâncias entre as sequências foram calculadas, podemos analisá-las graficamente através da árvore-guia gravada no arquivo .dnd. *ALERTA DE SPOILER*: este é o princípio da análise filogenética que veremos nas aulas seguintes.

**Atenção:** a árvore-guia serve apenas para definir a ordem em que as sequências são adicionadas ao alinhamento. Ela **não é uma filogenia**: só faz sentido interpretá-la evolutivamente quando as sequências são homólogas.

Vamos usar o Biopython para ler o arquivo .dnd e desenhar a árvore.

In [ ]:
tree = Phylo.read(dnd_file, "newick")
Phylo.draw(tree)

### Questão 3

**É possível extrair alguma informação a partir da árvore de distâncias acima? Compare os comprimentos dos ramos com os scores par a par exibidos pelo ClustalW.**

## 2. Para Ir Além (opcional)

### 2.1 Controle com sequências embaralhadas

Como saber se um score de alinhamento é alto ou baixo? Uma forma é comparar com o score obtido para sequências **aleatórias** de mesma composição. Abaixo, embaralhamos os resíduos de cada sequência (mantendo o tamanho e a composição de aminoácidos), alinhamos de novo e comparamos os scores par a par.

In [ ]:
import random
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

random.seed(42)
embaralhadas = []
for rec in SeqIO.parse(seq_file, "fasta"):
    residuos = list(str(rec.seq))
    random.shuffle(residuos)
    embaralhadas.append(SeqRecord(Seq("".join(residuos)), id=rec.id, description="embaralhada"))
SeqIO.write(embaralhadas, "Seqs0_embaralhadas.fasta", "fasta")

_, _, saida_emb = rodar_clustalw("Seqs0_embaralhadas.fasta")
reais = scores_par_a_par(saida)
aleatorios = scores_par_a_par(saida_emb)

nomes = [rec.id for rec in SeqIO.parse(seq_file, "fasta")]
print(f"{'Par de sequências':<55} {'Real':>5} {'Embaralhado':>12}")
for (i, j), score in reais.items():
    par = f"{nomes[i - 1]} x {nomes[j - 1]}"
    print(f"{par:<55} {score:5.0f} {aleatorios[(i, j)]:12.0f}")

### Questão Extra 1

**Os scores das sequências reais são muito diferentes dos scores das sequências embaralhadas? O que isso indica sobre a relação entre essas proteínas?**

### 2.2 Contraste com proteínas homólogas: globinas humanas

Agora vamos alinhar proteínas que **são** homólogas: as cadeias β (HBB), δ (HBD) e α (HBA) da hemoglobina e a mioglobina (MYG), todas humanas. As sequências são baixadas diretamente da API do UniProt.

In [ ]:
import urllib.request

acessos = ["P68871", "P02042", "P69905", "P02144"]  # HBB, HBD, HBA e MYG humanas
with open("globinas.fasta", "w") as saida_fasta:
    for acesso in acessos:
        url = f"https://rest.uniprot.org/uniprotkb/{acesso}.fasta"
        saida_fasta.write(urllib.request.urlopen(url).read().decode())

glob_aln, glob_dnd, saida_glob = rodar_clustalw("globinas.fasta")
mostrar_scores("globinas.fasta", saida_glob)
!cat {glob_aln}

In [ ]:
Phylo.draw(Phylo.read(glob_dnd, "newick"))

### Questão Extra 2

**A) Compare a linha de consenso (`*`, `:`, `.`) deste alinhamento com a do alinhamento da Seção 1. O que mudou?**

**B) Quais pares de globinas são mais parecidos? A árvore-guia faz sentido considerando o que você sabe sobre essas proteínas?**